# Version 2 Notebook 02:
# Two-Year Weather Training Request Plan

This notebook records the certified request plan for the
expanded weather only training period.

The weather only target period runs from 16 March 2024 to
15 March 2026. Forecast initialisations begin two days
earlier to support reconstruction of the earliest target
date.

The 00 and 12 UTC cycles are selected for retrieval. The
06 and 18 UTC cycles are documented as supplementary but
are not selected for bulk retrieval.

This notebook does not download forecasts, access market
prices or outcomes, fit a model, select calibration
parameters or calculate trading returns.

In [1]:
from pathlib import Path
import hashlib
import json

import pandas as pd


ROOT = Path.cwd().resolve()

for candidate in [ROOT, *ROOT.parents]:
    if (
        candidate
        / "config/v2/"
        "two_year_request_plan_spec.json"
    ).exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Repository root not found."
    )


def sha256(path: Path) -> str:
    return hashlib.sha256(
        path.read_bytes()
    ).hexdigest()


MANIFEST_PATH = (
    ROOT
    / "data/manifests/v2/"
    "02_two_year_request_plan_manifest.json"
)

MANIFEST = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

PLAN = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "02_two_year_request_plan.csv"
)

CORE = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "02_two_year_core_request_plan.csv"
)

SUPPLEMENTARY = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "02_two_year_supplementary_request_plan.csv"
)

SUPPORT = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "02_two_year_training_target_support_map.csv"
)

SUMMARY = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "02_two_year_request_plan_summary.csv"
)

CHECKS = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "02_two_year_request_plan_integrity_checks.csv"
)

In [2]:
print("=" * 76)
print("PHASE 3 TWO-YEAR WEATHER REQUEST PLAN")
print("=" * 76)

print(
    "Status:",
    MANIFEST["status"],
)

print(
    "Training period:",
    MANIFEST[
        "weather_only_training_start"
    ],
    "to",
    MANIFEST[
        "weather_only_training_end"
    ],
)

print(
    "Training dates:",
    MANIFEST[
        "weather_only_training_dates"
    ],
)

print(
    "Request initialisation dates:",
    MANIFEST[
        "request_initialisation_dates"
    ],
)

print(
    "Core cycles:",
    MANIFEST["core_cycles_utc"],
)

print(
    "Supplementary cycles:",
    MANIFEST[
        "supplementary_cycles_utc"
    ],
)

print(
    "Selected core requests:",
    MANIFEST[
        "selected_core_requests"
    ],
)

print(
    "Total documented requests:",
    MANIFEST[
        "total_planned_requests"
    ],
)

PHASE 3 TWO-YEAR WEATHER REQUEST PLAN
Status: TWO_YEAR_REQUEST_PLAN_CERTIFIED
Training period: 2024-03-16 to 2026-03-15
Training dates: 730
Request initialisation dates: 732
Core cycles: [0, 12]
Supplementary cycles: [6, 18]
Selected core requests: 1464
Total documented requests: 2928


In [3]:
print("Request-plan summary:")

print(
    SUMMARY.to_string(
        index=False
    )
)

print()
print("First eight selected requests:")

print(
    CORE[
        [
            "request_id",
            "run_init_utc",
            "cycle_utc",
            "run_period",
            "cache_path",
        ]
    ]
    .head(8)
    .to_string(
        index=False
    )
)

print()
print("First five target support rows:")

print(
    SUPPORT.head(5).to_string(
        index=False
    )
)

Request-plan summary:
           run_period    cycle_role  retrieval_selected  request_rows  request_dates  cycles
       support_buffer          core                True             4              2       2
       support_buffer supplementary               False             4              2       2
weather_only_training          core                True          1460            730       2
weather_only_training supplementary               False          1460            730       2

First eight selected requests:
     request_id              run_init_utc  cycle_utc            run_period                                                                     cache_path
ifs_20240314_00 2024-03-14T00:00:00+00:00          0        support_buffer data/raw/v2/open_meteo_single_runs_two_year/2024/03/ecmwf_ifs_20240314_00.json
ifs_20240314_12 2024-03-14T12:00:00+00:00         12        support_buffer data/raw/v2/open_meteo_single_runs_two_year/2024/03/ecmwf_ifs_20240314_12.json
ifs_20240315_00 202

In [4]:
required = CHECKS.loc[
    CHECKS["required"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin({"true", "1"})
].copy()

required_passed = (
    required["passed"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin({"true", "1"})
)

print("Required integrity checks:")

print(
    required.to_string(
        index=False
    )
)

assert required_passed.all()

assert (
    MANIFEST["phase_status"]
    == "PHASE3_COMPLETE"
)

assert (
    MANIFEST["status"]
    == "TWO_YEAR_REQUEST_PLAN_CERTIFIED"
)

assert (
    MANIFEST[
        "weather_only_training_dates"
    ]
    == 730
)

assert (
    MANIFEST[
        "request_initialisation_dates"
    ]
    == 732
)

assert (
    MANIFEST[
        "selected_core_requests"
    ]
    == 1464
)

assert (
    MANIFEST[
        "supplementary_planned_requests"
    ]
    == 1464
)

assert (
    MANIFEST[
        "total_planned_requests"
    ]
    == 2928
)

assert len(PLAN) == 2928
assert len(CORE) == 1464
assert len(SUPPLEMENTARY) == 1464
assert len(SUPPORT) == 730

assert PLAN["request_id"].is_unique
assert PLAN["run_init_utc"].is_unique

assert (
    sorted(
        CORE["cycle_utc"]
        .astype(int)
        .unique()
        .tolist()
    )
    == [0, 12]
)

assert (
    sorted(
        SUPPLEMENTARY[
            "cycle_utc"
        ]
        .astype(int)
        .unique()
        .tolist()
    )
    == [6, 18]
)

assert (
    SUPPORT[
        "support_complete"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin({"true", "1"})
    .all()
)

assert not MANIFEST[
    "network_requests_made"
]

assert not MANIFEST[
    "market_prices_accessed"
]

assert not MANIFEST[
    "realised_outcomes_accessed"
]

assert not MANIFEST[
    "model_fitted"
]

assert not MANIFEST[
    "model_selected"
]

for (
    relative_path,
    expected_hash,
) in MANIFEST[
    "output_hashes"
].items():
    assert (
        sha256(
            ROOT / relative_path
        )
        == expected_hash
    )

print()
print(
    "NOTEBOOK 02 REQUEST PLAN: PASSED"
)

print(
    "Next stage: retrieve the selected "
    "00 and 12 UTC requests."
)

Required integrity checks:
                                            check  required  passed                     detail
                                  phase2_complete      True    True            PHASE2_COMPLETE
                            phase2_pilot_approved      True    True SINGLE_RUNS_PILOT_APPROVED
                              training_date_count      True    True                        730
                               request_date_count      True    True                        732
                               total_request_rows      True    True                       2928
                       selected_core_request_rows      True    True                       1464
                       supplementary_request_rows      True    True                       1464
                               request_ids_unique      True    True                       2928
                       run_initialisations_unique      True    True                       2928
                       